# Load MAT-Appendix Results Without Retraining
This single cell downloads the latest reproducibility PKL from GitHub and reconstructs tables and a saved-XAI plot. It never trains MAT-Appendix.


In [ ]:
import sys,subprocess,requests,io,joblib,pandas as pd,numpy as np,matplotlib.pyplot as plt
subprocess.check_call([sys.executable,'-m','pip','install','-q','joblib','requests'])
BASE='https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/main/results'
r=requests.get(f'{BASE}/LATEST_RUN.txt',timeout=30)
if r.status_code!=200: raise RuntimeError('No completed GitHub run exists yet. Run MAT_Appendix_Primary_5Fold.ipynb first with GITHUB_TOKEN enabled.')
run_id=r.text.strip()
r=requests.get(f'{BASE}/runs/{run_id}/mat_appendix_reproducibility.pkl',timeout=180);r.raise_for_status()
bundle=joblib.load(io.BytesIO(r.content))
metrics=pd.DataFrame(bundle['metrics']['summary']);ci=pd.DataFrame(bundle['metrics']['ci']);fold_metrics=pd.DataFrame(bundle['metrics']['folds']);perm=pd.DataFrame(bundle['xai']['permutation'])
print('Loaded run:',run_id);print('Dataset SHA-256:',bundle['dataset']['sha256']);print('Proposed model fits saved:',len(bundle['folds']))
display(fold_metrics);display(metrics);display(ci);display(pd.DataFrame([bundle['xai']['status']]))
fig,ax=plt.subplots(figsize=(9,6));q=perm.head(20).sort_values('mean');ax.barh(q['feature'],q['mean']);ax.set_xlabel('Mean OOF PR-AUC decrease');ax.set_title('Saved MAT-Appendix permutation importance');plt.show()
saved_folds=bundle['folds'];saved_oof=bundle['oof'];saved_calibration=bundle['calibration'];saved_xai=bundle['xai']
print('Ready for future baseline/ablation notebooks: saved_folds, saved_oof, saved_calibration, saved_xai, bundle')
